Sascha Spors,
Professorship Signal Theory and Digital Signal Processing,
Institute of Communications Engineering (INT),
Faculty of Computer Science and Electrical Engineering (IEF),
University of Rostock,
Germany

# Data Driven Audio Signal Processing - A Tutorial with Computational Examples

Master Course #24512

- lecture: https://github.com/spatialaudio/data-driven-audio-signal-processing-lecture
- tutorial: https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise

Feel free to contact lecturer frank.schultz@uni-rostock.de

# PCA on Achieved Points of Written Examination 

In [ ]:
import numpy as np
import scipy
from scipy.linalg import svd, diagsvd
import matplotlib as mpl
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, sign=' ', suppress=True)

print(np.__version__)  # tested with 2.1.3
print(scipy.__version__)  # tested with 1.16.2
print(mpl.__version__)  # tested with 3.10.6

In [ ]:
X = np.loadtxt(open("exam_points_meanfree_unitvar.csv", "rb"),
               delimiter=";", skiprows=0)
N, F = X.shape
print(N, F)  # 34 students, 5 tasks
# written exam on Signals & Systems (a typical course in electrical engineering bachelor studies)
# the columns represent the original features and correspond to the following task topics:
task_label = ['Task 1: Convolution', 'Task 2: Fourier', 'Task 3: Sampling', 'Task 4: Laplace Domain', 'Task 5: z-Domain']

In [ ]:
# the data in exam_points_meanfree_unitvar.csv is already mean-free and columns have var=1
# so the numbers in X do not represent points or percentage,
# but rather encode the performance of the students per task in a normalised way
# X is however sorted: first row belongs to the best grade, last row to the worst grade
np.mean(X, axis=0), np.std(X, axis=0, ddof=1), np.var(X, axis=0, ddof=1)

In [ ]:
# prepare for PCA:
# make X zscore (although it is already)
mu = np.mean(X, axis=0)
X = X - mu  # mean free
sigma = np.sqrt(np.sum(X**2, axis=0) / (N-1))
X = X / sigma  # normalise to std=1
# normalise by number of examinations
Z = X / np.sqrt(N-1)
np.mean(X, axis=0), np.std(X, axis=0, ddof=1), np.var(X, axis=0, ddof=1)  # check

In [ ]:
# get covariance matrix
R = Z.T @ Z
R

In [ ]:
# PCA could go as an eigenwert problem of R
# i.e. spectral theorem as a special diagonalisation case
[Lmb, T] = np.linalg.eig(R)
Lmb = np.diag(Lmb)
Lmb, T  # for the PCA the eigvals need to be sorted in descending order
# this is typically not done in eig()
# in the current example and implementation
# they are however accidentally sorted

In [ ]:
np.linalg.trace(Lmb)  # total variance is 5 = 5 features (5 examination tasks)

In [ ]:
# T is an orthonormal basis
np.allclose(T.T@T, np.eye(F)), np.allclose(T@T.T, np.eye(F))
# R -> Lmb is a basis change with a PCA-dedicated, orthonormal basis T

In [ ]:
# PCA could go via SVD of Z
# sorting of singular values is then inherently done
# we use the economy SVD with full_matrices=False 
[U, s, Vh] = svd(Z, full_matrices=False)
V, S = Vh.T, diagsvd(s, F, F)

# switch polarities for nicer interpretation of the exam data
V[:,0] *= -1
U[:,0] *= -1

V[:,2] *= -1
U[:,2] *= -1

V[:,3] *= -1
U[:,3] *= -1

In [ ]:
# get PCA
PC_Features = U  # principal component scores (PCS) / a.k.a. principal component features (PCF)
PC_Loadings = V @ S  # principal components (PC) / a.k.a. principal component loadings
# cf. due to Z = U S V.T -> (S @ V.T).T
np.allclose(Z, PC_Features @ PC_Loadings.T)  # check correct SVD matrix factorisation
# note that other normalisation schemes exist for the PCA
# e.g. in data science / machine learning we typically see this:
# PC_Features = U @ S
# PC_Loadings = V
# then other post-processing / interpretation is needed

In [ ]:
# explained normalised variance is equal
# - to squared singular values of svd(Z)
# - to eigvals of eig(R)
variance = np.diag(S.T @ S)
variance, np.diag(Lmb)

In [ ]:
# total variance is 5 due to 5 features (examination tasks)
total_variance = np.sum(variance)
total_variance, F

In [ ]:
# cumsum variance in % 
cum_var = np.cumsum(variance) / total_variance * 100
cum_var

# Check via Plots

In [ ]:
plt.figure(figsize=(12,8))

plt.subplot(2,1,1)
for f in range(F):
    plt.plot(Z[:, f], 'o-', color='C'+str(f), label='Task '+str(f+1), ms=3)
plt.legend(loc='lower left')
plt.xticks([0, N-1], labels=['best grade', 'worst grade'])
plt.ylabel('original feature (column of Z)')
plt.grid(True)
plt.title(task_label)

plt.subplot(2,1,2)
for f in range(F):
    plt.plot(PC_Features[:, f], 'o-', color='C'+str(f), label='~PC '+str(f+1) + ', var='+ str('{0:.3f}'.format(variance[f])), lw=(F-f)*2/3, ms=(F-f)*3/2)
plt.legend(loc='lower left')
plt.xticks([0, N-1], labels=['best grade', 'worst grade'])
plt.ylabel('PC feature (column of U)')
plt.xlabel('examination index (sorted grade)')
plt.grid(True)
plt.title(['cum var in %:', cum_var])
plt.tight_layout()

In [ ]:
# correlation between original features (task) and PC features
pcf_label = ['PCF 1', 'PCF 2', 'PCF 3', 'PCF 4', 'PCF 5']
cmap = plt.get_cmap('Spectral_r', 8)
fig = plt.figure(figsize=(6,4))
ax = fig.add_subplot(111)
cax = ax.matshow(PC_Loadings, cmap=cmap, vmin=-1, vmax=+1)
fig.colorbar(cax)
ax.set_xticks(np.arange(len(pcf_label)))
ax.set_yticks(np.arange(len(task_label)))
ax.set_xticklabels(pcf_label)
ax.set_yticklabels(task_label)
ax.set_title('Loading Matrix = Task x loads to PC-Feature y')
plt.tight_layout()

# a rank 3 approximation of the data could be meaningful,
# i.e. using only PCF 1, PCF 2 and PCF 3 in the linear combination to reconstruct Z
# would only change one grade by a 1/3 grade step
# so 85.899 % explained variance would be enough to closely figure the actual grading

# PCF 1 and PCF 2 and PCF 3 might allow an intuitive interpretation:
# students are very well prepared to convolution, Laplace and z-Domain tasks
# as theses tasks are always very similar and definitely will be queried in the exam
# so, PCF 1 indidcates the performance on 'fulfilled' expectations and is
# highly correlated with the overall performance, and consequently with the achieved grade
# the Fourier task and the sampling task were chosen out of a wide range of options
# here students have rather 'unknown' expectations, which is why we need PCF 2 to cover this
# the convolution (+) and sampling (-) tasks load comparably high to PCF 3, so good performance
# on convolution comes along with bad performance on sampling for some students, especially those
# with an average grade (see the large peaks of the green curve in the middle of the above PC feature plot)
# PCF 4 and 5 show positive vs. negative correlations, i.e. mostly one good task vs. one bad task performance
# Some of these results are intuitive: we regularly observe that some students tend to have preferences
# for either the Laplace or the z-Domain task, thus the loadings onto PCF 5

In [ ]:
PC_Loadings  # the numbers for the above plot
# Task 1 loads onto PCF 1 with 0.819
# Task 4 loads onto PCF 1 with 0.859
# Task 5 loads onto PCF 1 with 0.853
# so they contribute with about equal importance
# to the PC feature 1
# PC feature 1 explains about 53.5 % of all variance

In [ ]:
PC_Loadings.T @ PC_Loadings  # eigvals of eig(R), squared singular vals of svd(Z) along the diagonal
# sorted from largest to smallest

In [ ]:
PC_Loadings**2  # squared loadings

In [ ]:
np.sum(PC_Loadings**2, axis=0)  # sum a column yields the variance of this PC


In [ ]:
np.sum(PC_Loadings**2, axis=1)  # communalities in one row sum up to 1

In [ ]:
variance, np.cumsum(variance)

## Copyright

- the notebooks are provided as [Open Educational Resources](https://en.wikipedia.org/wiki/Open_educational_resources)
- feel free to use the notebooks for your own purposes
- the text is licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/)
- the code of the IPython examples is licensed under the [MIT license](https://opensource.org/licenses/MIT)
- please attribute the work as follows: *Frank Schultz, Data Driven Audio Signal Processing - A Tutorial Featuring Computational Examples, University of Rostock* ideally with relevant file(s), github URL https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise, commit number and/or version tag, year.